# 04 — MICM Fuel Prices | Gold KPIs

KPIs **FUEL-001 to FUEL-012** per fuel and price week, into `gold.fuel_price_metrics`.

| KPI | Column | Definition |
|---|---|---|
| FUEL-001 | `price_dop` | Published price, DOP per gallon |
| FUEL-002 | `prev_week_price_dop` | Previous published week |
| FUEL-003 / 004 | `weekly_change_dop`, `weekly_change_pct` | vs previous published week |
| FUEL-005 | `monthly_change_pct` | vs last published week of the previous month |
| FUEL-006 | `yoy_change_pct` | vs latest week starting at least 364 days earlier (NULL if older than 371 days) |
| FUEL-007 | `ytd_change_pct` | vs last published week of the previous year |
| FUEL-008 / 009 | `ma4_price_dop`, `ma12_price_dop` | Trailing 4 / 12 published weeks |
| FUEL-010 / 011 | `high_52w_dop`, `low_52w_dop` | Trailing 364 days |
| FUEL-012 | `volatility_12w_pct` | Std. dev. of weekly % changes over the trailing 12 weeks (window set here) |

Weeks where a fuel was not published are excluded. Windows without enough history are NULL with `is_full_window_*` flags (same rule as FX decision D4). **Status:** Development.

In [0]:
import uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.silver.micm_fuel_prices_weekly"
TARGET = f"{CATALOG}.gold.fuel_price_metrics"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
VOL_WINDOW = 12
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

s = (spark.table(SOURCE).filter("is_published")
     .select("period_start_date", "period_end_date", "fuel_code", "fuel_name", F.col("price_dop").cast("double").alias("p")))
w = Window.partitionBy("fuel_code").orderBy("period_start_date")
day = F.datediff("period_start_date", F.lit("1970-01-01"))
w52 = Window.partitionBy("fuel_code").orderBy(day).rangeBetween(-363, 0)
first = Window.partitionBy("fuel_code")

g = (s.withColumn("prev", F.lag("p").over(w))
      .withColumn("ret", (F.col("p") / F.col("prev") - 1) * 100)
      .withColumn("ma4", F.avg("p").over(w.rowsBetween(-3, 0))).withColumn("n4", F.count("p").over(w.rowsBetween(-3, 0)))
      .withColumn("ma12", F.avg("p").over(w.rowsBetween(-11, 0))).withColumn("n12", F.count("p").over(w.rowsBetween(-11, 0)))
      .withColumn("vol", F.stddev("ret").over(w.rowsBetween(-(VOL_WINDOW - 1), 0))).withColumn("nret", F.count("ret").over(w.rowsBetween(-(VOL_WINDOW - 1), 0)))
      .withColumn("hi", F.max("p").over(w52)).withColumn("lo", F.min("p").over(w52))
      .withColumn("start0", F.min("period_start_date").over(first)))

# Reference prices: last week of previous month, previous year, and ~1 year earlier (as-of joins)
last_of = lambda key: (s.withColumn("k", key).withColumn("rn", F.row_number().over(Window.partitionBy("fuel_code", "k").orderBy(F.col("period_start_date").desc())))
                        .filter("rn = 1").select("fuel_code", "k", F.col("p").alias("ref")))
pm = last_of(F.date_format("period_start_date", "yyyy-MM"))
py = last_of(F.year("period_start_date").cast("string"))
g = (g.withColumn("k_m", F.date_format(F.add_months("period_start_date", -1), "yyyy-MM"))
      .join(pm.withColumnRenamed("k", "k_m").withColumnRenamed("ref", "ref_m"), ["fuel_code", "k_m"], "left")
      .withColumn("k_y", (F.year("period_start_date") - 1).cast("string"))
      .join(py.withColumnRenamed("k", "k_y").withColumnRenamed("ref", "ref_y"), ["fuel_code", "k_y"], "left"))
ya = s.select("fuel_code", F.col("period_start_date").alias("ya_date"), F.col("p").alias("ya_p"))
cand = (g.select("fuel_code", "period_start_date").join(ya, "fuel_code")
         .filter((F.datediff("period_start_date", "ya_date") >= 364) & (F.datediff("period_start_date", "ya_date") <= 371))
         .withColumn("rn", F.row_number().over(Window.partitionBy("fuel_code", "period_start_date").orderBy(F.col("ya_date").desc())))
         .filter("rn = 1").drop("rn"))
g = g.join(cand, ["fuel_code", "period_start_date"], "left")

r4 = lambda c: F.round(c, 4).cast("decimal(12,4)")
out = g.select(
    "period_start_date", "period_end_date", "fuel_code", "fuel_name",
    F.col("p").cast("decimal(10,2)").alias("price_dop"),
    F.col("prev").cast("decimal(10,2)").alias("prev_week_price_dop"),
    r4(F.col("p") - F.col("prev")).alias("weekly_change_dop"),
    r4(F.col("ret")).alias("weekly_change_pct"),
    r4((F.col("p") / F.col("ref_m") - 1) * 100).alias("monthly_change_pct"),
    r4((F.col("p") / F.col("ya_p") - 1) * 100).alias("yoy_change_pct"),
    r4((F.col("p") / F.col("ref_y") - 1) * 100).alias("ytd_change_pct"),
    F.when(F.col("n4") == 4, r4(F.col("ma4"))).alias("ma4_price_dop"),
    F.when(F.col("n12") == 12, r4(F.col("ma12"))).alias("ma12_price_dop"),
    (F.col("start0") <= F.date_sub("period_start_date", 363)).alias("is_full_window_52w"),
    F.when(F.col("start0") <= F.date_sub("period_start_date", 363), F.col("hi")).cast("decimal(10,2)").alias("high_52w_dop"),
    F.when(F.col("start0") <= F.date_sub("period_start_date", 363), F.col("lo")).cast("decimal(10,2)").alias("low_52w_dop"),
    F.when(F.col("nret") == VOL_WINDOW, r4(F.col("vol"))).alias("volatility_12w_pct"),
    F.lit("DOP per gallon").alias("price_unit"),
    F.year("period_start_date").alias("year"),
    F.lit(RUN_ID).alias("_gold_run_id"), F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"))

checks = []
def add(name, rtype, tested, failed):
    checks.append((RUN_ID, TARGET, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))
n, src = out.count(), s.count()
add("row_count_matches_published_silver", "enforced", n, abs(n - src))
add("unique_week_fuel", "enforced", n, n - out.select("period_start_date", "fuel_code").distinct().count())
add("price_within_52w_range", "enforced", n, out.filter((F.col("price_dop") < F.col("low_52w_dop")) | (F.col("price_dop") > F.col("high_52w_dop"))).count())
for c in checks: print(f"  {c[6]:<4} {c[2]:<36} tested={c[4]:,} failed={c[5]:,}")
assert not [c for c in checks if c[6] == "FAIL"]

out.orderBy("fuel_code", "period_start_date").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Gold: MICM fuel price KPIs FUEL-001 to FUEL-012. Grain: price week x fuel. DOP per gallon.'")
spark.createDataFrame(checks, "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

latest = spark.sql(f"""SELECT fuel_name, price_dop, weekly_change_pct, yoy_change_pct, ytd_change_pct FROM {TARGET}
  WHERE period_start_date = (SELECT MAX(period_start_date) FROM {TARGET}) AND fuel_code IN ('gasolina_premium','gasolina_regular','gasoil_optimo','gasoil_regular','glp') ORDER BY price_dop DESC""")
display(latest)
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "rows": n, "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks},
    "latest": [{k: str(v) for k, v in r.asDict().items()} for r in latest.collect()]}, ensure_ascii=False))